#### ABOUT THIS PROJECT
- Predict hourly electricity load (MW) for a region, and RAG-explains the drivers ("why is load spiking Friday evening? what's the weather effect?")
- Has a machine learning core, and a RAG that gives an evidence based answer.
- price forecasting (electricity spot price) using the same features — very current/valuable.


In [12]:
import pandas as pd
df = pd.read_csv("../data_raw/AEP_hourly.csv")
df.head()


,Datetime,AEP_MW
0,2004-12-31 01:00:00,13478.0
1,2004-12-31 02:00:00,12865.0
2,2004-12-31 03:00:00,12577.0
3,2004-12-31 04:00:00,12517.0
4,2004-12-31 05:00:00,12670.0


In [13]:
pd.read_csv("../data_raw/COMED_hourly.csv").head()


,Datetime,COMED_MW
0,2011-12-31 01:00:00,9970.0
1,2011-12-31 02:00:00,9428.0
2,2011-12-31 03:00:00,9059.0
3,2011-12-31 04:00:00,8817.0
4,2011-12-31 05:00:00,8743.0


In [14]:
import pandas as pd
import glob

# ------------------------------------------------------------------
# Build a long-form dataframe directly (one row per region+timestamp).
# 
# This replaces the old wide "outer/inner merge on Datetime" step,
# which either exploded (outer + DST fall-back -> 2^10 = 1024 rows at
# each 02:00 on 10 regions) or returned an EMPTY frame (inner, because
# the regions cover different date ranges so no timestamp exists in all
# 12 files). Building long-form avoids both problems entirely and is
# faster (no O(n^2) merge).
# ------------------------------------------------------------------

frames = []
for file in glob.glob("../data_raw/*_hourly.csv"):
    df = pd.read_csv(file, parse_dates=["Datetime"])

    # The only real column is the region's load, e.g. "AEP_MW".
    region = df.columns[1]

    # DST fall-back repeats the 02:00 wall-clock hour -> duplicate rows.
    # Keep just the first one so each timestamp is unique BEFORE any
    # shift/lag logic (otherwise lags point at fake "previous" hours).
    df = df.drop_duplicates(subset="Datetime")

    df = df.rename(columns={region: "MW"})
    # Strip the "_MW" suffix (was "-MW" before, which never matched).
    df["region"] = region.replace("_MW", "")
    df = df[["Datetime", "region", "MW"]]
    frames.append(df)

long_df = pd.concat(frames, ignore_index=True)
long_df["Datetime"] = pd.to_datetime(long_df["Datetime"])
long_df = long_df.sort_values(["region", "Datetime"]).reset_index(drop=True)
print("long_df shape:", long_df.shape)
print(long_df["region"].value_counts())
long_df.head()

long_df shape: (1090127, 3)
region
PJME        145362
PJMW        143202
DAYTON      121271
AEP         121269
DUQ         119064
DOM         116185
COMED        66493
FE           62870
NI           58450
DEOK         57735
EKPC         45330
PJM_Load     32896
Name: count, dtype: int64


,Datetime,region,MW
0,2004-10-01 01:00:00,AEP,12379.0
1,2004-10-01 02:00:00,AEP,11935.0
2,2004-10-01 03:00:00,AEP,11692.0
3,2004-10-01 04:00:00,AEP,11597.0
4,2004-10-01 05:00:00,AEP,11681.0


In [15]:
long_df.head()

,Datetime,region,MW
0,2004-10-01 01:00:00,AEP,12379.0
1,2004-10-01 02:00:00,AEP,11935.0
2,2004-10-01 03:00:00,AEP,11692.0
3,2004-10-01 04:00:00,AEP,11597.0
4,2004-10-01 05:00:00,AEP,11681.0


In [16]:
# The long_df is already clean (built in the cell above): one row per
# region+timestamp, no DST duplicates, no "_MW" suffix. Nothing more to
# drop or rename here — the old dropna/rename/sort logic lived here but
# is now handled during construction.
long_df = long_df.sort_values(["region", "Datetime"]).reset_index(drop=True)

In [17]:
long_df.head()

,Datetime,region,MW
0,2004-10-01 01:00:00,AEP,12379.0
1,2004-10-01 02:00:00,AEP,11935.0
2,2004-10-01 03:00:00,AEP,11692.0
3,2004-10-01 04:00:00,AEP,11597.0
4,2004-10-01 05:00:00,AEP,11681.0


In [18]:
df = long_df


In [19]:
def describer(df):
    print("-" *20 + " DataFrame Description " + "-" * 20)
    print("DataFrame Shape:", df.shape)
    print("\nDataFrame Info:")
    print(df.info())
    print("\nDataFrame Description:")
    print(df.describe())


describer(df)


-------------------- DataFrame Description --------------------
DataFrame Shape: (1090127, 3)

DataFrame Info:
<class 'pandas.DataFrame'>
RangeIndex: 1090127 entries, 0 to 1090126
Data columns (total 3 columns):
 #   Column    Non-Null Count    Dtype         
---  ------    --------------    -----         
 0   Datetime  1090127 non-null  datetime64[us]
 1   region    1090127 non-null  str           
 2   MW        1090127 non-null  float64       
dtypes: datetime64[us](1), float64(1), str(1)
memory usage: 29.1 MB
None

DataFrame Description:
                         Datetime            MW
count                     1090127  1.090127e+06
mean   2011-07-23 12:45:18.086241  1.120976e+04
min           1998-04-01 01:00:00  0.000000e+00
25%           2007-12-19 08:00:00  2.455000e+03
50%           2012-04-02 02:00:00  8.218000e+03
75%           2015-06-24 08:00:00  1.466500e+04
max           2018-08-03 00:00:00  6.200900e+04
std                           NaN  1.053449e+04


In [20]:
# create a hour column from the Datetime column
df['hour'] = df['Datetime'].dt.hour
# create a day_of_week column from the Datetime column
df['day_of_week'] = df['Datetime'].dt.dayofweek
# create month colunmn from the Datetime column
df['month'] = df['Datetime'].dt.month
# create a quarter column from the Datetime column
df['quarter'] = df['Datetime'].dt.quarter
# create a year column from the Datetime column
df['year'] = df['Datetime'].dt.year
# create an is weekend column from the Datetime column
df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)

import numpy as np

df['hour_sin'] = np.sin(2 * np.pi * df['hour']/24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour']/24)

df['day_of_week_sin'] = np.sin(2 * np.pi * df['day_of_week']/7)
df['day_of_week_cos'] = np.cos(2 * np.pi * df['day_of_week']/7)

df['month_sin'] = np.sin(2 * np.pi * df['month']/12)
df['month_cos'] = np.cos(2 * np.pi * df['month']/12)

# ------------------------------------------------------------------
# Lag features. Must be computed on a clean, continuous, deduplicated
# hourly series per region. The old code ran shift() on the raw df
# (which had DST dupes + gaps), so lags pointed at the wrong neighbor.
# We rebuild the full hourly grid per region, shift on that, then keep
# the lag value that exists at each ORIGINAL timestamp.
# ------------------------------------------------------------------
lag_cols = ["load_lag_1h", "load_lag_24h", "load_lag_168h", "load_lag_720h"]
df = df.sort_values(["region", "Datetime"])
for col in lag_cols:
    df[col] = float("nan")

for region, group in df.groupby("region", sort=False):
    idx = pd.date_range(group["Datetime"].min(), group["Datetime"].max(), freq="h")
    expanded = group.set_index("Datetime")["MW"].reindex(idx)
    for col in lag_cols:
        hours = int(col.split("_")[-1][:-1])  # "24h" -> 24
        lagged = expanded.shift(hours)
        df.loc[group.index, col] = lagged.reindex(group["Datetime"]).values

df['load_roll_mean_24h'] = df.groupby('region')['MW'].shift(1).rolling(24).mean()
df['load_roll_std_24h'] = df.groupby('region')['MW'].shift(1).rolling(24).std()
df['load_roll_mean_168h'] = df.groupby('region')['MW'].shift(1).rolling(168).mean()

import holidays as hd
us_holidays = hd.US()
df['is_holiday'] = df['Datetime'].dt.date.isin(us_holidays).astype(int)

# ------------------------------------------------------------------
# Extra features.
# is_month_start / is_month_end: cheap calendar flags (billing-cycle
# effects) that ML models often find useful.
# Fourier terms for the DAILY cycle, computed CORRECTLY per region:
#   t = hours since that region's first timestamp (not a global row
#   counter), and the divisor is 24 (hours/day), not SEASONAL_PERIODS.
#   We group per region, so each region's phase resets properly.
# ------------------------------------------------------------------
df['is_month_start'] = df['Datetime'].dt.is_month_start.astype(int)
df['is_month_end'] = df['Datetime'].dt.is_month_end.astype(int)

# complete the rolling set: 168h std (24h std already above)
df['load_roll_std_168h'] = df.groupby('region')['MW'].shift(1).rolling(168).std()

# daily-cycle Fourier terms, per region (uses the column name 'region')
df['fourier_sin_1'] = float('nan')
df['fourier_cos_1'] = float('nan')
df['fourier_sin_2'] = float('nan')
df['fourier_cos_2'] = float('nan')
for region, group in df.groupby('region', sort=False):
    t = (group['Datetime'] - group['Datetime'].min()).dt.total_seconds() / 3600.0  # hours since start
    df.loc[group.index, 'fourier_sin_1'] = np.sin(2 * np.pi * 1 * t / 24)
    df.loc[group.index, 'fourier_cos_1'] = np.cos(2 * np.pi * 1 * t / 24)
    df.loc[group.index, 'fourier_sin_2'] = np.sin(2 * np.pi * 2 * t / 24)
    df.loc[group.index, 'fourier_cos_2'] = np.cos(2 * np.pi * 2 * t / 24)


In [21]:
df.head()

,Datetime,region,MW,hour,day_of_week,month,quarter,year,is_weekend,hour_sin,...,load_roll_std_24h,load_roll_mean_168h,is_holiday,is_month_start,is_month_end,load_roll_std_168h,fourier_sin_1,fourier_cos_1,fourier_sin_2,fourier_cos_2
0,2004-10-01 01:00:00,AEP,12379.0,1,4,10,4,2004,0,0.258819,...,NaN,NaN,0,1,0,NaN,0.000000,1.000000,0.000000,1.000000e+00
1,2004-10-01 02:00:00,AEP,11935.0,2,4,10,4,2004,0,0.500000,...,NaN,NaN,0,1,0,NaN,0.258819,0.965926,0.500000,8.660254e-01
2,2004-10-01 03:00:00,AEP,11692.0,3,4,10,4,2004,0,0.707107,...,NaN,NaN,0,1,0,NaN,0.500000,0.866025,0.866025,5.000000e-01
3,2004-10-01 04:00:00,AEP,11597.0,4,4,10,4,2004,0,0.866025,...,NaN,NaN,0,1,0,NaN,0.707107,0.707107,1.000000,6.123234e-17
4,2004-10-01 05:00:00,AEP,11681.0,5,4,10,4,2004,0,0.965926,...,NaN,NaN,0,1,0,NaN,0.866025,0.500000,0.866025,-5.000000e-01


In [22]:
# save the new long_df to a CSV file
output_path = "../data_raw/merged_long_df.csv"
df.to_csv(output_path, index=False)
